In [ ]:
#!/usr/bin/env python3
"""
generate_london_grid_full.py

Generate a 50 m spacing point grid covering Greater London (full coverage, no max cap)
and save all lat,lon pairs to a CSV file.

Dependencies:
  pip install shapely pyproj requests numpy tqdm

Notes:
 - This will generate a LOT of points (hundreds of thousands). The script prints
   a rough estimate of grid cell count before iterating.
 - The script writes rows incrementally to avoid high memory usage.
"""

!pip install shapely pyproj requests numpy tqdm
import requests
import json
import csv
import math
import sys
import time
from shapely.geometry import shape, Point
from shapely.ops import transform
from pyproj import Transformer
import numpy as np
from tqdm import tqdm
import json

# -----------------------
# USER CONFIG
# -----------------------
PLACE_NAME = "Greater London"            # query for Nominatim
OUTPUT_CSV = "greater_london_grid_50m_full.csv"
SPACING_METERS = 50                      # grid spacing in meters
USER_AGENT = "grid-gen/1.0 (contact: you@example.com)"
# Choose projection for meter-accurate grid:
# Default uses Web Mercator (EPSG:3857). If you prefer British National Grid
# (EPSG:27700) for slightly better local accuracy inside UK, set USE_BNG = True.
USE_BNG = True
# -----------------------


def load_polygon_from_nominatim_json(path):
    """
    Load a Polygon or MultiPolygon from a saved Nominatim JSON response.
    Expects a list with at least one result containing a 'geojson' field.
    """
    with open(path, "r", encoding="utf-8") as fh:
        data = json.load(fh)

    if not isinstance(data, list) or not data:
        raise RuntimeError("Nominatim JSON must be a non-empty list")

    geojson = data[0].get("geojson")
    if geojson is None:
        raise RuntimeError("No 'geojson' field found in Nominatim JSON")

    if geojson.get("type") not in ("Polygon", "MultiPolygon"):
        raise RuntimeError(f"Unsupported geometry type: {geojson.get('type')}")

    return geojson


def fetch_place_geojson(place_name: str, user_agent: str = None):
    url = "https://nominatim.openstreetmap.org/search"
    headers = {"User-Agent": user_agent or USER_AGENT}
    params = {"q": place_name, "format": "json", "polygon_geojson": 1, "limit": 1}
    r = requests.get(url, params=params, headers=headers, timeout=60)
    r.raise_for_status()
    results = r.json()
    if not results:
        raise RuntimeError(f"No results from Nominatim for '{place_name}'")
    geojson = results[0].get("geojson")
    if not geojson:
        raise RuntimeError(f"Nominatim result for '{place_name}' has no geojson polygon")
    return geojson

def estimate_grid_count(bounds, spacing):
    minx, miny, maxx, maxy = bounds
    nx = math.ceil((maxx - minx) / spacing)
    ny = math.ceil((maxy - miny) / spacing)
    return nx * ny, nx, ny

def generate_grid_within_polygon(polygon_geojson, spacing_meters, out_csv_path, use_bng=False):
    """
    polygon_geojson: GeoJSON geometry (EPSG:4326)
    spacing_meters: grid spacing in meters
    Writes CSV with header "lat,lon".
    """
    poly_ll = shape(polygon_geojson)

    # choose projection
    if use_bng:
        # British National Grid (EPSG:27700) -- better local distance accuracy in UK
        to_m = Transformer.from_crs("EPSG:4326", "EPSG:27700", always_xy=True).transform
        to_ll = Transformer.from_crs("EPSG:27700", "EPSG:4326", always_xy=True).transform
        proj_name = "EPSG:27700 (BNG)"
    else:
        # Web Mercator, easy and available everywhere
        to_m = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True).transform
        to_ll = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True).transform
        proj_name = "EPSG:3857 (Web Mercator)"

    poly_m = transform(to_m, poly_ll)
    minx, miny, maxx, maxy = poly_m.bounds

    est_count, nx, ny = estimate_grid_count((minx, miny, maxx, maxy), spacing_meters)
    print(f"Projection: {proj_name}")
    print(f"Projected bounding box (meters): minx={minx:.1f}, miny={miny:.1f}, maxx={maxx:.1f}, maxy={maxy:.1f}")
    print(f"Rough grid cells: nx={nx}, ny={ny}, total ≈ {est_count:,}")
    print("This will test every grid cell and keep only points inside the polygon.")
    print(f"Output CSV: {out_csv_path}")
    print("Starting... (this may take several minutes depending on CPU and how many points are inside)")

    # Create coordinate arrays
    xs = np.arange(minx, maxx + spacing_meters, spacing_meters)
    ys = np.arange(miny, maxy + spacing_meters, spacing_meters)

    checked = 0
    saved = 0
    t0 = time.time()

    with open(out_csv_path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.writer(fh)
        writer.writerow(["lat", "lon"])

        # iterate rows for locality; show progress over rows
        with tqdm(total=len(ys), desc="rows", unit="row") as pbar_rows:
            for yi, y in enumerate(ys):
                # iterate x across this row
                for x in xs:
                    checked += 1
                    pt_m = Point(x, y)
                    if poly_m.contains(pt_m):
                        lon, lat = to_ll(x, y)
                        # write lat,lon with reasonable precision
                        writer.writerow([f"{lat:.7f}", f"{lon:.7f}"])
                        saved += 1
                pbar_rows.update(1)

    t1 = time.time()
    print(f"Done. points_checked={checked:,}, points_saved={saved:,}, elapsed={t1-t0:.1f}s")
    return out_csv_path, checked, saved

def main():
    print("=== Generate 50 m grid covering the whole Greater London polygon ===")
    print("Loading polygon from local Nominatim JSON file...")
    geojson = load_polygon_from_nominatim_json("london_coordinates.json")

    if geojson is None:
        raise SystemExit("Failed to fetch polygon geojson.")
    print("Polygon retrieved. Geometry type:", geojson.get("type"))

    out_csv, checked, saved = generate_grid_within_polygon(geojson, SPACING_METERS, OUTPUT_CSV, use_bng=USE_BNG)
    print(f"CSV saved to: {out_csv}")
    print("Finished.")

if __name__ == "__main__":
    main()


In [ ]:
# kepler_map.py
!pip install keplergl pandas
from keplergl import KeplerGl
import pandas as pd

CSV_PATH = "greater_london_grid_50m_full.csv"
OUTPUT_HTML = "london_kepler_map.html"

df = pd.read_csv(CSV_PATH)

map_1 = KeplerGl(height=800)
map_1.add_data(data=df, name="london_points")

config = {
    "version": "v1",
    "config": {
        "visState": {},
        "mapState": {
            "latitude": 51.5073509,   # London center latitude
            "longitude": -0.1277583,  # London center longitude
            "zoom": 11,               # starting zoom level
            "pitch": 0,
            "bearing": 0
        },
        "mapStyle": {}
    }
}
map_1.save_to_html(file_name=OUTPUT_HTML, config=config)
print("Saved kepler map to", OUTPUT_HTML)


In [ ]:
# folium_map.py
!pip install folium pandas tqdm
import pandas as pd
import folium
from folium.plugins import MarkerCluster
from IPython.display import IFrame, display  # for Colab/Jupyter

CSV_PATH = "greater_london_grid_50m_full.csv"   # file generated earlier
OUTPUT_HTML = "london_points_map.html"
START_LOCATION = [51.5074, -0.1278]        # London center (lat, lon)
START_ZOOM = 10

# optional: if your CSV contains a column 'blob_url' with publicly accessible URLs to images,
# folium popups will show a thumbnail and link to the full image.

df = pd.read_csv(CSV_PATH)

m = folium.Map(location=START_LOCATION, zoom_start=START_ZOOM, tiles="OpenStreetMap")

cluster = MarkerCluster(name="points", disableClusteringAtZoom=16).add_to(m)

for _, row in df.iterrows():
    lat = float(row["lat"])
    lon = float(row["lon"])
    # popup HTML: coordinates and optional image (if blob_url exists)
    html = f"<b>lat:</b> {lat:.6f}<br><b>lon:</b> {lon:.6f}"
    if "blob_url" in row and pd.notna(row["blob_url"]):
        img = row["blob_url"]
        # small thumbnail + link to open full image
        html += f"<br><a href='{img}' target='_blank'><img src='{img}' width='240'></a>"
    folium.Marker(location=[lat, lon], popup=folium.Popup(html, max_width=300)).add_to(cluster)

m.save(OUTPUT_HTML)
print("Saved map to", OUTPUT_HTML)

# In Colab / Jupyter you can display:
# display(IFrame(OUTPUT_HTML, width="100%", height=600))
